# accel-sim silicon anchor — gradient accumulation profiler

Times a real 4-layer chain (GPT-2's own shapes: 768→768→768→3072→768) at several
microbatch fractions, accumulating gradients across chunks before a would-be optimizer step,
and measures **peak activation memory** via `torch.cuda.max_memory_allocated()` — the
memory-vs-latency trade the `microbatch=` lever models.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Writes `gradaccum_profile.json`, prints it, and auto-downloads it. Bring that file back and run:

```bash
python validate/silicon/compare_gradaccum.py gradaccum_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
# NOTE: on a Colab T4 (Turing) or V100 use DTYPE = "fp16" -- those GPUs have
# NO bf16 tensor cores and bf16 silently falls back to a ~20x-slower kernel.
DTYPE   = "fp16"          # "bf16" | "fp16" | "fp32"
TOKENS  = 8 * 1024        # batch 8 x seq 1024
ITERS   = 30
WARMUP  = 10
OUT     = "gradaccum_profile.json"

# GPT-2's own shapes, chained sequentially (mirrors q_proj -> attn_out ->
# mlp_up -> mlp_down; matches simulator/workloads.py's dimensions).
CHAIN_DIMS = [(768, 768), (768, 768), (768, 3072), (3072, 768)]
FRACTIONS = [1, 2, 4, 8, 16]


In [ ]:
_DTYPES = {"bf16": "bfloat16", "fp16": "float16", "fp32": "float32"}

import torch
import torch.nn as nn
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = getattr(torch, _DTYPES[DTYPE])
M = TOKENS
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype={DTYPE}   tokens={M}   chain={CHAIN_DIMS}   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import statistics

def bench(frac):
    layers = [nn.Linear(k, n, bias=False).to(device=device, dtype=dtype)
              for k, n in CHAIN_DIMS]
    params = [p for l in layers for p in l.parameters()]
    x_full = torch.randn(M, CHAIN_DIMS[0][0], device=device, dtype=dtype)

    mb = max(1, M // frac)
    n_chunks = -(-M // mb)  # ceil

    def one_step():
        for p in params:
            p.grad = None
        for i in range(n_chunks):
            chunk = x_full[i * mb:(i + 1) * mb]
            if chunk.shape[0] == 0:
                continue
            out = chunk
            for l in layers:
                out = l(out)
            loss = out.float().square().mean() / n_chunks
            loss.backward()
        # optimizer step omitted -- optimizer_seconds is anchored separately
        # (profile_step.py); this isolates what microbatch= actually changes:
        # forward+backward cost and peak activation memory.

    times = []
    for i in range(WARMUP + ITERS):
        torch.cuda.synchronize()
        ev0, ev1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        ev0.record()
        one_step()
        ev1.record()
        torch.cuda.synchronize()
        if i >= WARMUP:
            times.append(ev0.elapsed_time(ev1))

    torch.cuda.reset_peak_memory_stats(device)
    one_step()
    torch.cuda.synchronize()
    peak_mb = torch.cuda.max_memory_allocated(device) / 1e6

    return {
        "mean_ms": statistics.fmean(times),
        "std_ms": statistics.pstdev(times) if len(times) > 1 else 0.0,
        "peak_activation_mb": peak_mb,
        "microbatch_tokens": mb,
        "n_chunks": n_chunks,
    }


In [ ]:
configs = {}
for frac in FRACTIONS:
    r = bench(frac)
    configs[str(frac)] = r
    print(f"  fraction {frac:3d}  (mb={r['microbatch_tokens']:5d}, n={r['n_chunks']:2d})  "
          f"step {r['mean_ms']:8.3f} ms   peak activations {r['peak_activation_mb']:8.1f} MB")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": _DTYPES[DTYPE], "tokens": M, "chain_dims": CHAIN_DIMS,
    "iters": ITERS, "warmup": WARMUP,
    "configs": configs, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
